# Part 2 - Realistic
Major difference with part 1 concern the distance constraint :

Distance constraints: To avoid over-concentration, any two facilities in the same area must be at least 0.06 miles apart. This means we cannot build new centers too close to existing ones or to each other within a zip code. We will use the provided potential_locations.csv to ensure new facilities are placed at valid locations respecting this distance.

Below is the Python script for Part 2. It reuses much of Part 1’s setup but adjusts expansion limits and costs, and incorporates the distance constraints. We also enforce that the number of new facilities built in each zip does not exceed the number of available potential locations that satisfy the 0.06 mile spacing (after filtering out those too close to existing centers). The model then finds the minimum funding required under these realistic conditions.

In [ ]:
import pandas as pd
import numpy as np
import math
import gurobipy as gp
from gurobipy import GRB
from sklearn.neighbors import BallTree

# 1. Load and preprocess
# --------------------------
population_df = pd.read_csv("./ChildCareDeserts_Data/population.csv")
income_df = pd.read_csv("./ChildCareDeserts_Data/avg_individual_income.csv").set_index(
    "ZIP code"
)
employment_df = pd.read_csv("./ChildCareDeserts_Data/employment_rate.csv").set_index(
    "zipcode"
)
childcare_df = pd.read_csv("./ChildCareDeserts_Data/child_care_regulated.csv")

location_df = pd.read_csv("./ChildCareDeserts_Data/potential_locations.csv")


population_df.set_index("zipcode", inplace=True)

# Compute number of children under 5 and number of children 0-12 in each ZIP.
population_df["children_under5"] = population_df["-5"]
population_df["children_0_12"] = (
    population_df["-5"] + population_df["5-9"] + 0.6 * population_df["10-14"]
)
population_df["children_0_12"] = population_df["children_0_12"].round()

population_df = population_df.join(income_df["average income"], how="left").join(
    employment_df["employment rate"], how="left"
)
population_df["average income"].fillna(
    population_df["average income"].median(), inplace=True
)
population_df["employment rate"].fillna(
    population_df["employment rate"].median(), inplace=True
)


population_df["high_demand"] = (population_df["employment rate"] >= 0.60) | (
    population_df["average income"] <= 60000
)

# 2. Determine current child care capacity per ZIP
# ------------------------------------------------
under5_caps = []
total_caps = []
for idx, row in childcare_df.iterrows():
    if row["children_capacity"] and row["children_capacity"] > 0:
        under5 = row["children_capacity"]
    else:
        under5 = (
            (0 if np.isnan(row["infant_capacity"]) else row["infant_capacity"])
            + (0 if np.isnan(row["toddler_capacity"]) else row["toddler_capacity"])
            + (0 if np.isnan(row["preschool_capacity"]) else row["preschool_capacity"])
        )
    if row["total_capacity"] and row["total_capacity"] > 0:
        total = row["total_capacity"]
    else:
        school = (
            0 if np.isnan(row["school_age_capacity"]) else row["school_age_capacity"]
        )
        total = under5 + school
    under5_caps.append(under5)
    total_caps.append(total)
childcare_df["under5_capacity"] = under5_caps
childcare_df["total_capacity_computed"] = total_caps

current_supply = childcare_df.groupby("zip_code")[
    ["under5_capacity", "total_capacity_computed"]
].sum()
current_supply.columns = ["current_under5_slots", "current_total_slots"]

population_df = population_df.join(current_supply, how="left")
population_df[["current_under5_slots", "current_total_slots"]] = population_df[
    ["current_under5_slots", "current_total_slots"]
].fillna(0)

# 3. Determine coverage requirements and identify child care deserts
# ------------------------------------------------------------------
population_df["required_total_slots"] = 0.33 * population_df["children_0_12"]
population_df.loc[population_df["high_demand"], "required_total_slots"] = (
    0.50 * population_df["children_0_12"]
)
population_df["required_under5_slots"] = 0.667 * population_df["children_under5"]
population_df["required_total_slots"] = np.ceil(population_df["required_total_slots"])
population_df["required_under5_slots"] = np.ceil(population_df["required_under5_slots"])

desert_mask = (
    population_df["current_total_slots"] <= population_df["required_total_slots"]
) | (population_df["current_under5_slots"] <= population_df["required_under5_slots"])
desert_zips = population_df.index[desert_mask]

# 4. Filter potential new facility locations by distance constraints
# ------------------------------------------------------------------
location_df = location_df[location_df["zipcode"].isin(desert_zips)].copy()

existing_coords = childcare_df[["latitude", "longitude"]].dropna().values
if len(existing_coords) > 0:
    existing_tree = BallTree(np.deg2rad(existing_coords), metric="haversine")
    cand_coords = np.deg2rad(location_df[["latitude", "longitude"]].values)
    dist, _ = existing_tree.query(cand_coords, k=1)
    dist_miles = dist.ravel() * 3959.0
    too_close_mask = dist_miles < 0.06
    location_df = location_df.loc[~too_close_mask].copy()

selected_sites = []
for zipcode, group in location_df.groupby("zipcode"):
    coords = group[["latitude", "longitude"]].values
    n = len(coords)
    if n == 0:
        continue
    sorted_idx = np.argsort(coords[:, 0])
    lat = coords[sorted_idx, 0]
    lon = coords[sorted_idx, 1]
    removed = np.zeros(n, dtype=bool)
    for i_idx in range(n):
        i = sorted_idx[i_idx]
        if removed[i]:
            continue
        selected_sites.append(group.index[i])
        lat_i = lat[i_idx]
        lon_i = lon[i_idx]
        for j_idx in range(i_idx + 1, n):
            j = sorted_idx[j_idx]
            if removed[j]:
                continue
            if lat[j_idx] - lat_i > 0.0009:
                break

            avg_lat = math.radians((lat_i + lat[j_idx]) / 2.0)
            d_lat = (lat[j_idx] - lat_i) * 69.0
            d_lon = (lon[j_idx] - lon_i) * (69.0 * math.cos(avg_lat))
            dist = math.hypot(d_lat, d_lon)
            if dist < 0.06:
                removed[j] = True

location_df = location_df.loc[selected_sites].copy()


# 5. Build the optimization model
# -------------------------------
model = gp.Model("ChildCareDesertElimination")
model.Params.OutputFlag = 0

expansion_vars = {}
new_vars = {}

# 5.1. Expansion variables for existing facilities in desert areas
for idx, fac in childcare_df.iterrows():
    zip_code = fac["zip_code"]
    if zip_code not in desert_zips:
        continue
    nf = fac["total_capacity_computed"]
    if nf <= 0:
        continue

    exp_vars = {}
    exp_vars["x_total"] = model.addVar(
        vtype=GRB.INTEGER, lb=0, ub=math.floor(0.20 * nf)
    )
    exp_vars["x_u5"] = model.addVar(vtype=GRB.INTEGER, lb=0, ub=math.floor(0.20 * nf))
    exp_vars["y1"] = model.addVar(name=f"y1_{idx}", lb=0, ub=0.10 * nf)
    exp_vars["y2"] = model.addVar(name=f"y2_{idx}", lb=0, ub=0.05 * nf)
    exp_vars["y3"] = model.addVar(name=f"y3_{idx}", lb=0, ub=0.05 * nf)
    expansion_vars[idx] = exp_vars

    model.addConstr(
        exp_vars["x_total"] == exp_vars["y1"] + exp_vars["y2"] + exp_vars["y3"]
    )
    model.addConstr(exp_vars["x_u5"] <= exp_vars["x_total"])
    model.addConstr(exp_vars["y1"] <= 0.10 * nf)
    model.addConstr(exp_vars["y2"] <= 0.05 * nf)
    model.addConstr(exp_vars["y3"] <= 0.05 * nf)

# 5.2. New facility variables for candidate sites in desert areas
for idx, site in location_df.iterrows():
    zip_code = site["zipcode"]
    if zip_code not in desert_zips:
        continue
    var_small = model.addVar(vtype=GRB.BINARY, name=f"new_small_{idx}")
    var_med = model.addVar(vtype=GRB.BINARY, name=f"new_medium_{idx}")
    var_large = model.addVar(vtype=GRB.BINARY, name=f"new_large_{idx}")
    new_vars[idx] = {
        "small": var_small,
        "medium": var_med,
        "large": var_large,
        "zip": zip_code,
    }
    model.addConstr(var_small + var_med + var_large <= 1)

# 6. Coverage constraints for each desert ZIP code
# ------------------------------------------------
for zip_code in desert_zips:
    req_total = float(population_df.loc[zip_code, "required_total_slots"])
    req_under5 = float(population_df.loc[zip_code, "required_under5_slots"])
    curr_total = float(population_df.loc[zip_code, "current_total_slots"])
    curr_under5 = float(population_df.loc[zip_code, "current_under5_slots"])

    lhs_total = gp.LinExpr()
    lhs_under5 = gp.LinExpr()
    for f_idx, exp_vars in expansion_vars.items():
        if childcare_df.loc[f_idx, "zip_code"] != zip_code:
            continue
        lhs_total += exp_vars["x_total"]
        lhs_under5 += exp_vars["x_u5"]
    for i, vars_dict in new_vars.items():
        if vars_dict["zip"] != zip_code:
            continue
        lhs_total += (
            100 * vars_dict["small"]
            + 200 * vars_dict["medium"]
            + 400 * vars_dict["large"]
        )
        lhs_under5 += (
            50 * vars_dict["small"]
            + 100 * vars_dict["medium"]
            + 200 * vars_dict["large"]
        )

    additional_total_needed = max(0, math.ceil(req_total - curr_total) + 1)
    additional_under5_needed = max(0, math.ceil(req_under5 - curr_under5) + 1)
    model.addConstr(
        lhs_total >= additional_total_needed, name=f"TotalCapacityReq_{zip_code}"
    )
    model.addConstr(
        lhs_under5 >= additional_under5_needed, name=f"Under5CapacityReq_{zip_code}"
    )

# 7. Objective: Minimize total cost
# ---------------------------------
obj = gp.LinExpr()

# 7.1 Add expansion costs:
for f_idx, exp_vars in expansion_vars.items():
    nf = childcare_df.loc[f_idx, "total_capacity_computed"]
    if nf > 0:
        cost_per_slot_seg1 = (20000 + 200 * nf) / nf
        cost_per_slot_seg2 = (20000 + 400 * nf) / nf
        cost_per_slot_seg3 = (20000 + 1000 * nf) / nf
    else:
        cost_per_slot_seg1 = cost_per_slot_seg2 = cost_per_slot_seg3 = 0
    obj += cost_per_slot_seg1 * exp_vars["y1"]
    obj += cost_per_slot_seg2 * exp_vars["y2"]
    obj += cost_per_slot_seg3 * exp_vars["y3"]
    obj += 100.0 * exp_vars["x_u5"]

# 7.2 Add new facility construction costs:
for i, vars_dict in new_vars.items():
    obj += 70000.0 * vars_dict["small"]
    obj += 105000.0 * vars_dict["medium"]
    obj += 135000.0 * vars_dict["large"]

model.setObjective(obj, GRB.MINIMIZE)

# 8. Optimize the model
model.optimize()

# 9. Output the minimum total funding required
min_cost = model.ObjVal
min_cost = math.ceil(min_cost)
print(f"Min cost:${min_cost:,.2f}")

/var/folders/j5/g36rlrm17vv157kr1c8_ndyw0000gn/T/ipykernel_54991/306928679.py:41: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  population_df["average income"].fillna(
/var/folders/j5/g36rlrm17vv157kr1c8_ndyw0000gn/T/ipykernel_54991/306928679.py:44: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behave

Set parameter Username
Set parameter LicenseID to value 2722177
Academic license - for non-commercial use only - expires 2026-10-14
Min cost:$391,626,999.00
